# Estate Monitor: collector
Polls the Fabric and Power BI **admin APIs** and streams what changed to the Real-Time Estate Monitor. It fills the gaps that native Fabric events (job, workspace item and capacity events from Real-Time hub) don't cover:

| Collector | Source API | Event types | Default cadence |
|---|---|---|---|
| activity | Power BI activity log (`admin/activityevents`) | `Estate.Activity` (views, exports, Copilot, sharing, deployments) | 2 min |
| refreshes | `admin/capacities/refreshables` (+ optional per-workspace history) | `Estate.Refresh` | 5 min |
| inventory | Fabric `admin/workspaces` and `admin/items` (all item types, including reports and semantic models) | `Estate.Inventory.*` created / updated / deleted | 15 min |
| access | `admin/groups?$expand=users` | `Estate.Access.*` granted / role changed / removed | 30 min |
| capacities | `admin/capacities` | `Estate.Capacity.*` | 30 min |

Every pass also sends `Estate.Collector.Heartbeat` (or `.Error`) events so the dashboard shows collector health.

**Run it**
- Attach a **default lakehouse** (state file, and the `activity_events` Delta table that `Fabric_Governance` uses for stale scoring).
- Run as a **Fabric administrator**, or set `AUTH_MODE = "service_principal"` with a Key Vault secret.
- Start with `SINK = "dryrun"` and `RUN_MINUTES = 0` to check API access, then switch to `"eventstream"`.
- Schedule it **hourly** with `RUN_MINUTES = 55`: each run keeps polling for 55 minutes, so data arrives within a couple of minutes.
- Only changes are sent. The first run sends a baseline of every workspace, item and role assignment.
- Admin APIs are rate limited (for example about 200 activity-log calls an hour). Keep the cadences at or above the defaults on large tenants.


## Parameters

In [ ]:
# ---- Where events go ----
SINK = "eventstream"                 # "eventstream" (custom endpoint), "eventhouse" (direct to KQL) or "dryrun" (print only)
EVENTSTREAM_CONNECTION_STRING = ""   # Eventstream > custom endpoint > Event Hub > SAS Key > connection string. Prefer Key Vault:
KEY_VAULT_URI = ""                   # e.g. "https://my-vault.vault.azure.net/"
EVENTSTREAM_SECRET_NAME = ""         # Key Vault secret that holds the connection string
EVENTHOUSE_QUERY_URI = ""            # SINK = "eventhouse": KQL database > Query URI
KQL_DATABASE = "EstateMonitor"
KQL_TABLE = "RawEvents"

# ---- Identity ----
AUTH_MODE = "user"                   # "user" = whoever runs or owns the schedule (must be a Fabric administrator)
TENANT_ID = ""                       # AUTH_MODE = "service_principal" only; the app needs the tenant setting
CLIENT_ID = ""                       #   "Service principals can access read-only admin APIs"
CLIENT_SECRET_NAME = ""              #   secret name in KEY_VAULT_URI. Never paste a secret into the notebook.

# ---- What to collect, and how often (minutes). 0 turns a collector off ----
COLLECT_EVERY_MIN = {
    "activity":   2,    # activity log: views, exports, Copilot, sharing, deployments
    "refreshes":  5,    # semantic model refreshes (Fabric job events cover pipelines, notebooks, etc.)
    "inventory": 15,    # every workspace and item, with created / updated / deleted detection
    "access":    30,    # workspace role grants, changes and removals
    "capacities": 30,   # capacity SKU and state
}
RUN_MINUTES = 55                     # keep polling for this long; schedule the notebook hourly. 0 = one pass
POLL_SECONDS = 60
ACTIVITY_FIRST_RUN_LOOKBACK_MIN = 120
ACTIVITY_OVERLAP_MIN = 45            # re-read this window every pass to catch late-arriving activity events
INCLUDE_PERSONAL = False             # include "My workspace" items in the inventory
REFRESH_DETAIL_WORKSPACES = []       # optional: workspace names to read full refresh history (identity needs access)
MASK_USERS = False                   # True = replace user names with a stable hash (for screen sharing a real tenant)
MASK_SALT = "change-me"

# ---- Lakehouse outputs (attach a default lakehouse) ----
WRITE_ACTIVITY_TABLE = True          # also keep the activity log in a Delta table for Fabric_Governance (stale scoring)
ACTIVITY_TABLE = "activity_events"
STATE_PATH = "/lakehouse/default/Files/estate_monitor/collector_state.json"


## Helpers: identity, event envelope and senders

In [ ]:
# ---- Shared helpers: identity, REST calls, event envelope, sinks ----
import json, time, uuid, hmac, hashlib, base64, urllib.parse, os, copy, random, math
from datetime import datetime, timezone, timedelta
import requests

try:
    nbu = notebookutils                      # Fabric runtime global
except NameError:                            # older runtimes
    from notebookutils import mssparkutils as nbu

def utcnow():
    return datetime.now(timezone.utc)

def iso(dt):
    return dt.astimezone(timezone.utc).strftime("%Y-%m-%dT%H:%M:%S.%f")[:-3] + "Z"

def _secret(name):
    if not name:
        return ""
    if not KEY_VAULT_URI:
        raise ValueError(f"Set KEY_VAULT_URI to read the secret '{name}'.")
    return nbu.credentials.getSecret(KEY_VAULT_URI, name)

class Identity:
    """Tokens for the Fabric, Power BI and Kusto APIs, as the notebook user or a service principal."""
    SCOPES = {"pbi": "https://analysis.windows.net/powerbi/api/.default",
              "fabric": "https://api.fabric.microsoft.com/.default"}
    def __init__(self):
        self._cache = {}
        self._app = None
        if AUTH_MODE == "service_principal":
            import msal
            secret = _secret(CLIENT_SECRET_NAME)
            self._app = msal.ConfidentialClientApplication(
                CLIENT_ID, authority=f"https://login.microsoftonline.com/{TENANT_ID}", client_credential=secret)

    def token(self, resource):
        hit = self._cache.get(resource)
        if hit and hit[1] > time.time() + 120:
            return hit[0]
        if self._app is None:
            if resource == "kusto":
                try:
                    tok = nbu.credentials.getToken("kusto")
                except Exception:
                    tok = nbu.credentials.getToken(EVENTHOUSE_QUERY_URI.rstrip("/"))
            else:
                tok = nbu.credentials.getToken("pbi")      # accepted by the Power BI and Fabric APIs
        else:
            scope = self.SCOPES.get(resource) or f"{EVENTHOUSE_QUERY_URI.rstrip('/')}/.default"
            res = self._app.acquire_token_for_client(scopes=[scope])
            if "access_token" not in res:
                raise RuntimeError(f"Token request failed: {res.get('error_description', res)}")
            tok = res["access_token"]
        self._cache[resource] = (tok, time.time() + 600)
        return tok

def make_event(etype, data, subject="", event_id=None, when=None, capacity_id="", source="estate-monitor/collector"):
    return {"specversion": "1.0", "id": event_id or str(uuid.uuid4()), "source": source, "type": etype,
            "subject": subject, "time": when or iso(utcnow()), "dataschemaversion": "1.0",
            "capacityId": capacity_id or "", "data": data}

class EventstreamSink:
    """Sends events to an Eventstream custom endpoint (Event Hub-compatible) over HTTPS with a SAS key.
    No extra packages needed."""
    MAX_BATCH_BYTES = 900_000
    MAX_BATCH_EVENTS = 500
    def __init__(self, connection_string):
        parts = dict(p.split("=", 1) for p in connection_string.strip().strip(";").split(";") if "=" in p)
        self.host = parts["Endpoint"].replace("sb://", "").strip("/")
        self.entity = parts["EntityPath"]
        self.key_name = parts["SharedAccessKeyName"]
        self.key = parts["SharedAccessKey"]
        self.url = f"https://{self.host}/{self.entity}/messages?timeout=60&api-version=2014-01"

    def _sas(self, ttl=3600):
        uri = urllib.parse.quote_plus(f"https://{self.host}/{self.entity}")
        expiry = str(int(time.time() + ttl))
        sig = base64.b64encode(hmac.new(self.key.encode("utf-8"), f"{uri}\n{expiry}".encode("utf-8"), hashlib.sha256).digest())
        return f"SharedAccessSignature sr={uri}&sig={urllib.parse.quote_plus(sig)}&se={expiry}&skn={self.key_name}"

    def _post(self, batch):
        body = json.dumps([{"Body": json.dumps(e, default=str)} for e in batch])
        for attempt in range(5):
            r = requests.post(self.url, data=body.encode("utf-8"), timeout=60,
                              headers={"Authorization": self._sas(),
                                       "Content-Type": "application/vnd.microsoft.servicebus.json"})
            if r.status_code in (200, 201):
                return
            if r.status_code in (429, 500, 503) and attempt < 4:
                time.sleep(2 ** attempt); continue
            raise RuntimeError(f"Eventstream send failed: HTTP {r.status_code} {r.text[:300]}")

    def send(self, events):
        batch, size = [], 2
        for e in events:
            n = len(json.dumps(e, default=str)) + 16
            if batch and (size + n > self.MAX_BATCH_BYTES or len(batch) >= self.MAX_BATCH_EVENTS):
                self._post(batch); batch, size = [], 2
            batch.append(e); size += n
        if batch:
            self._post(batch)
        return len(events)

class EventhouseSink:
    """Writes events straight into the RawEvents table with the Spark Kusto connector (no Eventstream)."""
    COLS = ["id", "source", "type", "subject", "time", "specversion", "dataschemaversion", "capacityId", "data"]
    def __init__(self, identity):
        if not EVENTHOUSE_QUERY_URI:
            raise ValueError("Set EVENTHOUSE_QUERY_URI for SINK = 'eventhouse'.")
        self.identity = identity
    def send(self, events):
        if not events:
            return 0
        from pyspark.sql.types import StructType, StructField, StringType
        rows = [tuple(json.dumps(e["data"], default=str) if c == "data" else str(e.get(c) or "") for c in self.COLS) for e in events]
        df = spark.createDataFrame(rows, StructType([StructField(c, StringType()) for c in self.COLS]))
        (df.write.format("com.microsoft.kusto.spark.synapse.datasource")
           .option("kustoCluster", EVENTHOUSE_QUERY_URI)
           .option("kustoDatabase", KQL_DATABASE)
           .option("kustoTable", KQL_TABLE)
           .option("accessToken", self.identity.token("kusto"))
           .option("adjustSchema", "GenerateDynamicCsvMapping")
           .mode("Append").save())
        return len(events)

class DryRunSink:
    """Prints a sample instead of sending. Use SINK = 'dryrun' to test API access first."""
    def __init__(self): self.sent = []
    def send(self, events):
        self.sent.extend(events)
        for e in events[:3]:
            print("  sample:", json.dumps(e, default=str)[:300])
        return len(events)

def build_sink(identity):
    if SINK == "eventstream":
        cs = EVENTSTREAM_CONNECTION_STRING or _secret(EVENTSTREAM_SECRET_NAME)
        if not cs:
            raise ValueError("Set EVENTSTREAM_CONNECTION_STRING, or KEY_VAULT_URI + EVENTSTREAM_SECRET_NAME.")
        return EventstreamSink(cs)
    if SINK == "eventhouse":
        return EventhouseSink(identity)
    if SINK == "dryrun":
        return DryRunSink()
    raise ValueError("SINK must be 'eventstream', 'eventhouse' or 'dryrun'.")


## Collectors

In [ ]:
FABRIC = "https://api.fabric.microsoft.com/v1/"
PBI = "https://api.powerbi.com/v1.0/myorg/"
identity = Identity()
sink = build_sink(identity)

def api_get(url, resource="pbi", params=None, ok404=False):
    for attempt in range(6):
        r = requests.get(url, params=params, timeout=60,
                         headers={"Authorization": f"Bearer {identity.token(resource)}"})
        if r.status_code == 429 or r.status_code >= 500:
            wait = int(r.headers.get("Retry-After", 2 ** (attempt + 2)))
            print(f"  throttled or busy ({r.status_code}), waiting {wait}s")
            time.sleep(min(wait, 120)); continue
        if ok404 and r.status_code == 404:
            return None
        if not r.ok:
            raise RuntimeError(f"GET {url.split('?')[0]} failed: HTTP {r.status_code} {r.text[:300]}")
        return r.json()
    raise RuntimeError(f"GET {url} kept failing after retries")

def fabric_list(path, key):
    rows, token = [], None
    while True:
        j = api_get(FABRIC + path, "fabric", params={"continuationToken": token} if token else None)
        rows += j.get(key, []) or []
        token = j.get("continuationToken")
        if not token:
            return rows

def mask(user):
    if not MASK_USERS or not user:
        return user
    return "user-" + hashlib.sha256((MASK_SALT + str(user).lower()).encode()).hexdigest()[:10]

def h(obj):
    return hashlib.sha256(json.dumps(obj, sort_keys=True, default=str).encode()).hexdigest()[:16]

# ---------------- state ----------------
def load_state():
    try:
        with open(STATE_PATH, encoding="utf-8") as f:
            return json.load(f)
    except FileNotFoundError:
        return {}
    except Exception as e:
        print("State file unreadable, starting fresh:", e)
        return {}

def save_state(st):
    os.makedirs(os.path.dirname(STATE_PATH), exist_ok=True)
    tmp = STATE_PATH + ".tmp"
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(st, f)
    os.replace(tmp, STATE_PATH)

# ---------------- collectors: each returns a list of events and updates `st` in place ----------------
def collect_capacities(st):
    caps = api_get(PBI + "admin/capacities").get("value", [])
    prev, now_map, events = st.get("capacities", {}), {}, []
    for c in caps:
        rec = {"capacityId": c.get("id"), "capacityName": c.get("displayName"), "capacitySku": c.get("sku"),
               "state": c.get("state"), "region": c.get("region"),
               "admins": ";".join(mask(a) for a in (c.get("admins") or []))}
        hv = h(rec); now_map[c["id"]] = {"h": hv, "n": rec["capacityName"]}
        if c["id"] not in prev:
            events.append(make_event("Estate.Capacity.Discovered", rec, f"/capacities/{c['id']}", capacity_id=c["id"]))
        elif prev[c["id"]]["h"] != hv:
            events.append(make_event("Estate.Capacity.Changed", rec, f"/capacities/{c['id']}", capacity_id=c["id"]))
    st["capacities"] = now_map
    return events

def collect_inventory(st):
    workspaces = fabric_list("admin/workspaces", "workspaces")
    items = fabric_list("admin/items", "itemEntities")
    ws_by_id = {w["id"]: w for w in workspaces}
    first_run = "items" not in st
    events = []

    # workspaces
    prev_ws, now_ws = st.get("workspaces", {}), {}
    for w in workspaces:
        if not INCLUDE_PERSONAL and str(w.get("type", "")).lower() == "personal":
            continue
        rec = {"workspaceId": w["id"], "workspaceName": w.get("name"), "workspaceType": w.get("type"),
               "state": w.get("state"), "capacityId": w.get("capacityId")}
        hv = h(rec); now_ws[w["id"]] = {"h": hv, "n": w.get("name"), "r": rec}
        if first_run or w["id"] not in prev_ws:
            kind = "WorkspaceDiscovered" if first_run else "WorkspaceCreated"
        elif prev_ws[w["id"]]["h"] != hv:
            kind = "WorkspaceUpdated"
            rec["changedFields"] = ",".join(k for k in rec if prev_ws[w["id"]].get("r", {}).get(k) != rec[k])
        else:
            continue
        events.append(make_event(f"Estate.Inventory.{kind}", rec, f"/workspaces/{w['id']}", capacity_id=w.get("capacityId")))

    # items
    prev_items, now_items = st.get("items", {}), {}
    for it in items:
        w = ws_by_id.get(it.get("workspaceId"), {})
        if not INCLUDE_PERSONAL and str(w.get("type", "")).lower() == "personal":
            continue
        creator = it.get("creatorPrincipal") or {}
        rec = {"itemId": it["id"], "itemName": it.get("name"), "itemKind": it.get("type"),
               "workspaceId": it.get("workspaceId"), "workspaceName": w.get("name"), "workspaceType": w.get("type"),
               "capacityId": it.get("capacityId") or w.get("capacityId"), "state": it.get("state"),
               "lastUpdatedDate": it.get("lastUpdatedDate"),
               "createdBy": mask((creator.get("userDetails") or {}).get("userPrincipalName") or creator.get("displayName"))}
        hv = h(rec)
        now_items[it["id"]] = {"h": hv, "r": {k: rec[k] for k in ("itemName", "itemKind", "workspaceId", "workspaceName", "state", "lastUpdatedDate")}}
        old = prev_items.get(it["id"])
        if first_run or old is None:
            kind = "ItemDiscovered" if first_run else "ItemCreated"
        elif old["h"] != hv:
            kind = "ItemUpdated"
            rec["changedFields"] = ",".join(k for k, v in now_items[it["id"]]["r"].items() if old.get("r", {}).get(k) != v)
        else:
            continue
        events.append(make_event(f"Estate.Inventory.{kind}", rec, f"/workspaces/{rec['workspaceId']}/items/{it['id']}",
                                 capacity_id=rec["capacityId"]))

    # deletions: only trust them when the scan looks complete
    if not first_run and now_items and len(now_items) >= 0.5 * len(prev_items):
        for iid, old in prev_items.items():
            if iid not in now_items:
                r = old.get("r", {})
                events.append(make_event("Estate.Inventory.ItemDeleted", {"itemId": iid, **r},
                                         f"/workspaces/{r.get('workspaceId')}/items/{iid}"))
        for wid, old in prev_ws.items():
            if wid not in now_ws:
                events.append(make_event("Estate.Inventory.WorkspaceDeleted", old.get("r", {"workspaceId": wid}),
                                         f"/workspaces/{wid}"))
    st["workspaces"], st["items"] = now_ws, now_items
    return events

def collect_access(st):
    groups, skip = [], 0
    while True:
        j = api_get(PBI + "admin/groups", params={"$top": 5000, "$skip": skip, "$expand": "users",
                                                  "$filter": "type eq 'Workspace'"})
        batch = j.get("value", []); groups += batch
        if len(batch) < 5000:
            break
        skip += 5000
    first_run = "access" not in st
    prev, now_map, events = st.get("access", {}), {}, []
    for g in groups:
        for u in g.get("users") or []:
            principal = u.get("emailAddress") or u.get("identifier") or u.get("displayName")
            key = f"{g['id']}|{str(principal).lower()}"
            role = u.get("groupUserAccessRight")
            rec = {"workspaceId": g["id"], "workspaceName": g.get("name"), "principal": mask(principal),
                   "principalType": u.get("principalType"), "role": role}
            now_map[key] = {"role": role, "r": rec}
            old = prev.get(key)
            if first_run:
                events.append(make_event("Estate.Access.Discovered", rec, f"/workspaces/{g['id']}"))
            elif old is None:
                events.append(make_event("Estate.Access.Granted", rec, f"/workspaces/{g['id']}"))
            elif old["role"] != role:
                events.append(make_event("Estate.Access.RoleChanged", {**rec, "previousRole": old["role"]}, f"/workspaces/{g['id']}"))
    if not first_run and now_map:
        for key, old in prev.items():
            if key not in now_map:
                events.append(make_event("Estate.Access.Removed", {**old["r"], "previousRole": old["role"]},
                                         f"/workspaces/{old['r'].get('workspaceId')}"))
    st["access"] = now_map
    return events

ACTIVITY_DELTA_ROWS = []

def collect_activity(st):
    end = utcnow() - timedelta(minutes=1)
    wm = st.get("activity_watermark")
    start = (datetime.fromisoformat(wm.replace("Z", "+00:00")) - timedelta(minutes=ACTIVITY_OVERLAP_MIN)) if wm \
            else end - timedelta(minutes=ACTIVITY_FIRST_RUN_LOOKBACK_MIN)
    seen = st.get("activity_seen", {})
    events, raw = [], []
    s = start
    while s < end:                                   # the API needs start and end on the same UTC day
        day_end = datetime(s.year, s.month, s.day, tzinfo=timezone.utc) + timedelta(days=1) - timedelta(milliseconds=1)
        e = min(end, day_end)
        url = (PBI + "admin/activityevents?startDateTime='" + iso(s) + "'&endDateTime='" + iso(e) + "'")
        while url:
            j = api_get(url)
            raw += j.get("activityEventEntities", []) or []
            url = None if j.get("lastResultSet", True) else j.get("continuationUri")
        s = e + timedelta(milliseconds=1)
    for a in raw:
        aid = a.get("Id") or a.get("id")
        if not aid or aid in seen:
            continue
        seen[aid] = a.get("CreationTime")
        item_id = a.get("ArtifactId") or a.get("ReportId") or a.get("DatasetId") or a.get("DashboardId") or a.get("ObjectId")
        data = {"activity": a.get("Activity"), "operation": a.get("Operation"), "userId": mask(a.get("UserId")),
                "userType": str(a.get("UserType", "")), "workload": a.get("Workload"),
                "workspaceId": a.get("WorkspaceId"), "workspaceName": a.get("WorkSpaceName") or a.get("WorkspaceName"),
                "itemId": item_id,
                "itemName": a.get("ArtifactName") or a.get("ItemName") or a.get("ReportName") or a.get("DatasetName") or a.get("DashboardName"),
                "itemKind": a.get("ArtifactKind") or a.get("ObjectType"),
                "consumptionMethod": a.get("ConsumptionMethod"), "distributionMethod": a.get("DistributionMethod"),
                "isSuccess": a.get("IsSuccess"), "capacityId": a.get("CapacityId"), "capacityName": a.get("CapacityName"),
                "requestId": a.get("RequestId")}
        events.append(make_event("Estate.Activity", data, f"/workspaces/{data['workspaceId']}/items/{item_id}",
                                 event_id=aid, when=a.get("CreationTime"), capacity_id=a.get("CapacityId")))
        if WRITE_ACTIVITY_TABLE:
            ACTIVITY_DELTA_ROWS.append({
                "Id": aid, "CreationTime": a.get("CreationTime"), "Activity": a.get("Activity"), "Operation": a.get("Operation"),
                "UserId": mask(a.get("UserId")), "UserType": str(a.get("UserType", "")), "Workload": a.get("Workload"),
                "WorkspaceId": a.get("WorkspaceId"), "WorkSpaceName": data["workspaceName"],
                "ArtifactId": a.get("ArtifactId"), "ArtifactName": a.get("ArtifactName"), "ArtifactKind": a.get("ArtifactKind"),
                "ItemId": a.get("ItemId"), "ItemName": a.get("ItemName"), "ReportId": a.get("ReportId"),
                "ReportName": a.get("ReportName"), "DatasetId": a.get("DatasetId"), "DatasetName": a.get("DatasetName"),
                "DashboardId": a.get("DashboardId"), "ObjectId": a.get("ObjectId"), "CapacityId": a.get("CapacityId"),
                "CapacityName": a.get("CapacityName"), "ConsumptionMethod": a.get("ConsumptionMethod"),
                "DistributionMethod": a.get("DistributionMethod"), "IsSuccess": str(a.get("IsSuccess")),
                "RawJson": json.dumps(a, default=str)})
    # forget ids older than the re-read window
    cutoff = iso(start - timedelta(hours=2))
    st["activity_seen"] = {k: v for k, v in seen.items() if (v or "") >= cutoff}
    st["activity_watermark"] = iso(end)
    return events

def _refresh_event(ds_id, ds_name, ws_id, ws_name, ref, extra=None):
    err = ""
    sej = ref.get("serviceExceptionJson")
    if sej:
        try:
            ej = json.loads(sej); err = ej.get("errorDescription") or ej.get("errorCode") or sej
        except Exception:
            err = sej
    data = {"jobInstanceId": ref.get("requestId") or f"{ds_id}|{ref.get('startTime')}", "itemId": ds_id,
            "itemName": ds_name, "itemKind": "SemanticModel", "workspaceId": ws_id, "workspaceName": ws_name,
            "jobType": "Refresh", "jobInvokeType": ref.get("refreshType"), "jobStatus": ref.get("status"),
            "jobStartTime": ref.get("startTime"), "jobEndTime": ref.get("endTime"), "errorMessage": str(err)[:2000]}
    if extra:
        data.update(extra)
    return data

def collect_refreshes(st):
    seen = st.get("refresh_seen", {})
    events, found = [], []
    skip = 0
    while True:                                       # tenant-wide: last refresh of everything on a capacity
        j = api_get(PBI + "admin/capacities/refreshables",
                    params={"$top": 1000, "$skip": skip, "$expand": "capacity,group"})
        vals = j.get("value", []) or []
        for r in vals:
            lr = r.get("lastRefresh") or {}
            if lr:
                g = r.get("group") or {}
                found.append(_refresh_event(r.get("id"), r.get("name"), g.get("id"), g.get("name"), lr,
                                            {"refreshCount": r.get("refreshCount"), "refreshFailures": r.get("refreshFailures")}))
        if len(vals) < 1000:
            break
        skip += 1000
    if REFRESH_DETAIL_WORKSPACES:                     # optional: full recent history for chosen workspaces
        for name in REFRESH_DETAIL_WORKSPACES:
            gs = api_get(PBI + "groups", params={"$filter": f"name eq '{name}'"}).get("value", [])
            for g in gs:
                for d in api_get(PBI + f"groups/{g['id']}/datasets").get("value", []):
                    hist = api_get(PBI + f"groups/{g['id']}/datasets/{d['id']}/refreshes", params={"$top": 5}, ok404=True) or {}
                    for ref in hist.get("value", []):
                        found.append(_refresh_event(d["id"], d.get("name"), g["id"], g.get("name"), ref))
    for data in found:
        key = f"{data['itemId']}|{data['jobInstanceId']}|{data['jobStatus']}"
        if key in seen:
            continue
        seen[key] = data.get("jobStartTime") or iso(utcnow())
        events.append(make_event("Estate.Refresh", data,
                                 f"/workspaces/{data['workspaceId']}/items/{data['itemId']}/jobs/instances/{data['jobInstanceId']}",
                                 when=data.get("jobEndTime") or data.get("jobStartTime")))
    if len(seen) > 50000:                             # keep the newest keys
        seen = dict(sorted(seen.items(), key=lambda kv: kv[1] or "")[-40000:])
    st["refresh_seen"] = seen
    return events

COLLECTORS = {"activity": collect_activity, "refreshes": collect_refreshes, "inventory": collect_inventory,
              "access": collect_access, "capacities": collect_capacities}


In [ ]:
def write_activity_table(rows):
    """Append new activity rows to the lakehouse Delta table, skipping Ids already there."""
    if not rows:
        return 0
    from pyspark.sql.types import StructType, StructField, StringType
    cols = list(rows[0].keys())
    df = spark.createDataFrame([tuple(None if r.get(c) in (None, "") else str(r.get(c)) for c in cols) for r in rows],
                               StructType([StructField(c, StringType()) for c in cols])).dropDuplicates(["Id"])
    if not spark.catalog.tableExists(ACTIVITY_TABLE):
        df.write.format("delta").saveAsTable(ACTIVITY_TABLE)
        return df.count()
    df.createOrReplaceTempView("_new_activity")
    spark.sql(f"MERGE INTO {ACTIVITY_TABLE} t USING _new_activity s ON t.Id = s.Id WHEN NOT MATCHED THEN INSERT *")
    return df.count()


## Poll and send

In [ ]:
def run_cycle(st, last_run):
    """One polling pass: run the collectors that are due, send, then commit state."""
    work = copy.deepcopy(st)
    events, beats = [], []
    ACTIVITY_DELTA_ROWS.clear()
    for name, every in COLLECT_EVERY_MIN.items():
        if not every or time.time() - last_run.get(name, 0) < every * 60:
            continue
        t0 = time.time()
        try:
            ev = COLLECTORS[name](work)
            events += ev
            beats.append(make_event("Estate.Collector.Heartbeat",
                                    {"collector": name, "eventsSent": len(ev), "durationSec": round(time.time() - t0, 1)}))
            print(f"  {name}: {len(ev)} events in {time.time() - t0:.1f}s")
        except Exception as e:
            beats.append(make_event("Estate.Collector.Error", {"collector": name, "message": str(e)[:2000],
                                                               "durationSec": round(time.time() - t0, 1)}))
            print(f"  {name}: FAILED {e}")
            work[name + "_failed_at"] = iso(utcnow())
        last_run[name] = time.time()
    sent = sink.send(events + beats)                  # raises on failure, so state is not committed
    if WRITE_ACTIVITY_TABLE and ACTIVITY_DELTA_ROWS:
        try:
            print(f"  activity table: {write_activity_table(ACTIVITY_DELTA_ROWS)} new rows")
        except Exception as e:
            print("  activity table write failed (events were still sent):", e)
    if SINK != "dryrun":
        save_state(work)
    return work, sent

state = load_state()
last_run = {}
deadline = time.time() + RUN_MINUTES * 60
cycle = 0
while True:
    cycle += 1
    print(f"Cycle {cycle} at {iso(utcnow())}")
    try:
        state, sent = run_cycle(state, last_run)
        print(f"  sent {sent} events")
    except Exception as e:
        print("  send failed; this pass will be retried next cycle:", e)
        last_run.clear()
    if RUN_MINUTES <= 0 or time.time() + POLL_SECONDS >= deadline:
        break
    time.sleep(POLL_SECONDS)
print("Collector finished.")
